# 04 · 模型模块（Models）功能演示

演示 Boosting/经典模型统一接口、逻辑回归、标准/取整/概率评分卡、评分漂移校准与自定义损失训练。

In [1]:
import warnings, os
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import hscredit

# 路径约定：从 notebooks/ 目录运行，数据在 ../examples，产物输出到 model_report/
DATA = os.path.join("..", "examples", "hscredit_yyp.xlsx")
if not os.path.exists(DATA):
    DATA = os.path.join("examples", "hscredit_yyp.xlsx")
OUT = "model_report"
os.makedirs(OUT, exist_ok=True)

df = pd.read_excel(DATA)
df["放款时间"] = pd.to_datetime(df["放款时间"])
y = df["FPD"].astype(int)
NUM_FEATURES = ["珊瑚92", "青云24", "衡枢鉴真分老客版", "占信V3", "天创小额网贷分", "近六个月非银多头机构数"]
CAT_FEATURE = "商品类别"
print("数据形状:", df.shape)
print("坏样本率: {:.4f}".format(y.mean()))
df.head()

数据形状: (970, 18)
坏样本率: 0.1402


,客户编号,放款时间,放款金额,商品类别,MOB1,CURRENT_DPD,中智小牛分C3,珊瑚92,极光欺诈分6v1,青云24,占信V3,轻花老客海纳子分V1,天创小额网贷分,近六个月非银多头机构数,手机号近一个月非银多头机构数,身份证近一个月非银多头机构数,衡枢鉴真分老客版,FPD
0,1985945640026276096,2026-02-03,1399,礼包,0,0,NaN,NaN,NaN,656,NaN,NaN,630,51,15,15,0.0242,0
1,1985972188268592896,2026-02-04,1399,礼包,0,0,NaN,NaN,NaN,565,NaN,NaN,583,56,6,18,0.0492,0
2,1986034700861140992,2025-11-06,3960,珠宝首饰,0,0,NaN,NaN,NaN,708,NaN,NaN,764,68,17,20,0.0546,0
3,1986264852923760896,2025-11-06,3960,珠宝首饰,0,0,NaN,NaN,NaN,555,NaN,NaN,712,45,15,15,0.0899,0
4,1986265696509906944,2026-01-26,1399,礼包,0,0,NaN,NaN,NaN,581,NaN,NaN,641,67,32,32,0.0678,0


## 1. 准备训练/测试集与 WOE 特征

In [2]:
from sklearn.model_selection import train_test_split
from hscredit.core.binning import OptimalBinning

X = df[NUM_FEATURES]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
Xtr_raw, Xte_raw = Xtr.fillna(0), Xte.fillna(0)

binner = OptimalBinning(method='best_iv', max_n_bins=5).fit(Xtr, ytr)
Xtr_woe = binner.transform(Xtr, metric='woe')
Xte_woe = binner.transform(Xte, metric='woe')
print('训练集:', Xtr.shape, ' 测试集:', Xte.shape)

训练集: (679, 6)  测试集: (291, 6)


## 2. Boosting 与经典模型统一接口（fit/predict_proba/evaluate）

In [3]:
from hscredit.core.models import (XGBoostRiskModel, LightGBMRiskModel, CatBoostRiskModel, NGBoostRiskModel,
    RandomForestRiskModel, ExtraTreesRiskModel, GradientBoostingRiskModel)

model_makers = {
    'XGBoost': lambda: XGBoostRiskModel(n_estimators=80),
    'LightGBM': lambda: LightGBMRiskModel(n_estimators=80),
    'CatBoost': lambda: CatBoostRiskModel(n_estimators=80),
    'NGBoost': lambda: NGBoostRiskModel(n_estimators=80),
    'RandomForest': lambda: RandomForestRiskModel(),
    'ExtraTrees': lambda: ExtraTreesRiskModel(),
    'GradientBoosting': lambda: GradientBoostingRiskModel(),
}
rows = []
for name, mk in model_makers.items():
    m = mk(); m.fit(Xtr_raw, ytr)
    ev = m.evaluate(Xte_raw, yte)
    rows.append({'模型': name, 'KS': round(ev.get('ks', ev.get('KS', np.nan)), 4), 'AUC': round(ev.get('auc', ev.get('AUC', np.nan)), 4)})
model_compare = pd.DataFrame(rows)
model_compare

[1]	valid_0's binary_logloss: 0.400834
[2]	valid_0's binary_logloss: 0.400496
[3]	valid_0's binary_logloss: 0.401491
[4]	valid_0's binary_logloss: 0.404469
[5]	valid_0's binary_logloss: 0.404561
[6]	valid_0's binary_logloss: 0.408595
[7]	valid_0's binary_logloss: 0.411011
[8]	valid_0's binary_logloss: 0.408343
[9]	valid_0's binary_logloss: 0.408216
[10]	valid_0's binary_logloss: 0.408512
[11]	valid_0's binary_logloss: 0.411112
[12]	valid_0's binary_logloss: 0.411328
[13]	valid_0's binary_logloss: 0.412782
[14]	valid_0's binary_logloss: 0.41812
[15]	valid_0's binary_logloss: 0.418991
[16]	valid_0's binary_logloss: 0.419647
[17]	valid_0's binary_logloss: 0.417506
[18]	valid_0's binary_logloss: 0.422248
[19]	valid_0's binary_logloss: 0.423713
[20]	valid_0's binary_logloss: 0.427029
[21]	valid_0's binary_logloss: 0.427101
[22]	valid_0's binary_logloss: 0.427206
[23]	valid_0's binary_logloss: 0.432482
[24]	valid_0's binary_logloss: 0.431186
[25]	valid_0's binary_logloss: 0.430733
[26]	valid

,模型,KS,AUC
0,XGBoost,0.1095,0.5262
1,LightGBM,0.1125,0.5346
2,CatBoost,0.2122,0.5947
3,NGBoost,0.1882,0.5918
4,RandomForest,0.2345,0.6179
5,ExtraTrees,0.1969,0.6167
6,GradientBoosting,0.1548,0.5760


## 3. 逻辑回归 + 系数显著性 summary

In [4]:
from hscredit.core.models import LogisticRegression
lr = LogisticRegression(); lr.fit(Xtr_woe, ytr)
lr.summary()

,Coef.,Std.Err,z,P>|z|,[0.025,0.975],VIF
const,-1.8045,0.1202,-15.0080,0.0000,-2.0402,-1.5689,1.0774
珊瑚92,0.8174,0.3973,2.0576,0.0396,0.0388,1.5961,1.0271
青云24,0.2650,0.3392,0.7813,0.4346,-0.3999,0.9299,1.1367
衡枢鉴真分老客版,0.7314,0.2193,3.3353,0.0009,0.3016,1.1613,1.0666
占信V3,0.8009,0.2285,3.5050,0.0005,0.3530,1.2488,1.0273
天创小额网贷分,0.5289,0.3861,1.3699,0.1707,-0.2278,1.2855,1.0626
近六个月非银多头机构数,0.6935,0.3879,1.7879,0.0738,-0.0668,1.4539,1.0178


## 4. 标准评分卡 ScoreCard（Score = A - B·ln(odds)）

In [5]:
from hscredit.core.models import ScoreCard
sc = ScoreCard(); sc.fit(Xtr_woe, ytr)
scores = sc.predict_score(Xte_woe)
print('测试集评分范围: [{:.1f}, {:.1f}]'.format(scores.min(), scores.max()))
sc.scorecard_scale().head(12)

测试集评分范围: [421.0, 2152.1]


,刻度项,刻度值,备注
0,base_odds,35,好坏比（好:坏），内部换算实际 odds = 1/base_odds
1,base_score,750,基础 odds 对应的分数
2,rate,2,odds 增加的倍率
3,pdo,60,odds 增加 2 倍时分数变化量
4,B (pdo/ln(rate)),86.5617,pdo / ln(2)
5,A (offset),442.2430,base_score + B * ln(实际odds)，实际odds 见 base_odds 备注
6,formula,Score = 442.243 - 86.5617 × ln(odds),评分卡转换公式，odds = P(坏) / P(好)（同 score_formula 的「公式」）


## 5. 评分卡刻度表（scorecard_points）与评分→坏率表

In [6]:
points = sc.scorecard_points()
display(points.head(12))
sc.score_to_bad_rate_table(scores, yte.values, n_bins=10)

,变量名称,变量含义,变量分箱,对应分数,WOE值
0,基础分,截距项（基准分数）,-,598.4449,NaN
1,珊瑚92,,WOE: -1.8475,130.7283,-1.8475
2,珊瑚92,,WOE: -0.5819,41.1721,-0.5819
3,珊瑚92,,WOE: -0.4353,30.7987,-0.4353
4,珊瑚92,,WOE: 0.0731,-5.1692,0.0731
5,珊瑚92,,WOE: 0.0767,-5.4291,0.0767
6,珊瑚92,,WOE: 0.7174,-50.7627,0.7174
7,青云24,,WOE: -0.5258,12.0623,-0.5258
8,青云24,,WOE: -0.2359,5.4112,-0.2359
9,青云24,,WOE: 0.2756,-6.3223,0.2756


,评分区间,样本数,坏样本数,坏样本率,好样本数,Odds,累计好样本占比,累计坏样本占比,KS
0,"(421.02930000000003, 527.7594]",31,8,25.81%,23,2.88,0.0920,0.1951,0.1031
1,"(527.7594, 556.0282]",28,7,25.00%,21,3.00,0.1760,0.3659,0.1899
2,"(556.0282, 572.7738]",33,4,12.12%,29,7.25,0.2920,0.4634,0.1714
3,"(572.7738, 598.9182]",25,2,8.00%,23,11.50,0.3840,0.5122,0.1282
4,"(598.9182, 613.3551]",29,4,13.79%,25,6.25,0.4840,0.6098,0.1258
5,"(613.3551, 627.1109]",29,2,6.90%,27,13.50,0.5920,0.6585,0.0665
6,"(627.1109, 652.341]",32,4,12.50%,28,7.00,0.7040,0.7561,0.0521
7,"(652.341, 687.5801]",28,4,14.29%,24,6.00,0.8000,0.8537,0.0537
8,"(687.5801, 759.4243]",27,4,14.81%,23,5.75,0.8920,0.9512,0.0592
9,"(759.4243, 2152.1339]",29,2,6.90%,27,13.50,1.0000,1.0000,0.0000


## 6. 取数原因（reason code，评分卡可解释性）

In [7]:
sc.get_reason(Xte_woe.head(5), keep=3)

,reason
0,天创小额网贷分(降低28.6分); 青云24(降低15.1分); 近六个月非银多头机构数(提升0.0分)
1,占信V3(提升42.6分); 珊瑚92(提升36.2分); 衡枢鉴真分老客版(提升25.2分)
2,近六个月非银多头机构数(降低70.3分); 天创小额网贷分(降低28.6分); 青云24(降低15.1分)
3,近六个月非银多头机构数(降低70.3分); 青云24(降低15.1分); 天创小额网贷分(提升0.0分)
4,青云24(提升6.7分); 珊瑚92(提升0.3分); 近六个月非银多头机构数(提升0.0分)


## 7. 取整评分卡 RoundScoreCard 与原始评分卡一致性

In [8]:
from hscredit.core.models import RoundScoreCard
rsc = RoundScoreCard(); rsc.fit(Xtr_woe, ytr)
cmp = pd.DataFrame({'标准评分': sc.predict_score(Xte_woe.head(8)), '取整评分': rsc.predict_score(Xte_woe.head(8))})
cmp

,标准评分,取整评分
0,569.4487,569.4300
1,705.4234,705.4200
2,499.1714,499.1500
3,527.7594,527.7400
4,620.0062,619.9900
5,638.5852,638.5700
6,687.5801,687.5700
7,623.2669,623.2500


## 8. 概率评分卡 ProbabilityScoreCard（任意模型概率→评分）

In [9]:
from hscredit.core.models import ProbabilityScoreCard
base_model = LightGBMRiskModel(n_estimators=80).fit(Xtr_raw, ytr)
psc = ProbabilityScoreCard(model=base_model); psc.fit(Xtr_raw, ytr)
psc.predict_score(Xte_raw)[:8]

[1]	valid_0's binary_logloss: 0.405984
[2]	valid_0's binary_logloss: 0.406395
[3]	valid_0's binary_logloss: 0.40534
[4]	valid_0's binary_logloss: 0.406335
[5]	valid_0's binary_logloss: 0.408527
[6]	valid_0's binary_logloss: 0.404865
[7]	valid_0's binary_logloss: 0.40341
[8]	valid_0's binary_logloss: 0.404987
[9]	valid_0's binary_logloss: 0.407949
[10]	valid_0's binary_logloss: 0.410344
[11]	valid_0's binary_logloss: 0.404966
[12]	valid_0's binary_logloss: 0.403915
[13]	valid_0's binary_logloss: 0.407788
[14]	valid_0's binary_logloss: 0.406952
[15]	valid_0's binary_logloss: 0.406856
[16]	valid_0's binary_logloss: 0.410379
[17]	valid_0's binary_logloss: 0.411845
[18]	valid_0's binary_logloss: 0.41356
[19]	valid_0's binary_logloss: 0.412838
[20]	valid_0's binary_logloss: 0.418265
[21]	valid_0's binary_logloss: 0.422124
[22]	valid_0's binary_logloss: 0.426817
[23]	valid_0's binary_logloss: 0.428157
[24]	valid_0's binary_logloss: 0.42671
[25]	valid_0's binary_logloss: 0.432058
[26]	valid_0'

array([542., 704., 579., 599., 554., 605., 669., 543.])

## 9. 评分漂移校准 ScoreDriftCalibrator

In [10]:
from hscredit.core.models import ScoreDriftCalibrator
cal = ScoreDriftCalibrator(); cal.fit(sc, Xte_woe, X_reference=Xtr_woe)
calibrated = cal.predict_score(Xte_woe)
print('校准后评分均值: {:.2f}'.format(np.mean(calibrated)))

校准后评分均值: 0.14


## 10. 自定义损失函数训练（Focal Loss 处理不平衡）

In [11]:
from hscredit.core.models.losses import FocalLoss
m_focal = LightGBMRiskModel(objective=FocalLoss(), n_estimators=80); m_focal.fit(Xtr_raw, ytr)
ev = m_focal.evaluate(Xte_raw, yte)
print('FocalLoss 模型 KS={:.4f}'.format(ev.get('ks', ev.get('KS', np.nan))))

[1]	valid_0's binary_logloss: 4.80054
[2]	valid_0's binary_logloss: 4.77667
[3]	valid_0's binary_logloss: 4.75346
[4]	valid_0's binary_logloss: 4.73105
[5]	valid_0's binary_logloss: 4.70944
[6]	valid_0's binary_logloss: 4.68869
[7]	valid_0's binary_logloss: 4.66855
[8]	valid_0's binary_logloss: 4.64904
[9]	valid_0's binary_logloss: 4.63026
[10]	valid_0's binary_logloss: 4.61212
[11]	valid_0's binary_logloss: 4.59466
[12]	valid_0's binary_logloss: 4.57776
[13]	valid_0's binary_logloss: 4.56138
[14]	valid_0's binary_logloss: 4.54562
[15]	valid_0's binary_logloss: 4.53035
[16]	valid_0's binary_logloss: 4.51615
[17]	valid_0's binary_logloss: 4.50178
[18]	valid_0's binary_logloss: 4.4889
[19]	valid_0's binary_logloss: 4.47588
[20]	valid_0's binary_logloss: 4.46376
[21]	valid_0's binary_logloss: 4.45122
[22]	valid_0's binary_logloss: 4.43963
[23]	valid_0's binary_logloss: 4.42733
[24]	valid_0's binary_logloss: 4.41576
[25]	valid_0's binary_logloss: 4.40469
[26]	valid_0's binary_logloss: 4.39

[48]	valid_0's binary_logloss: 4.22719
[49]	valid_0's binary_logloss: 4.22115
[50]	valid_0's binary_logloss: 4.21603
[51]	valid_0's binary_logloss: 4.21139
[52]	valid_0's binary_logloss: 4.20652
[53]	valid_0's binary_logloss: 4.20273
[54]	valid_0's binary_logloss: 4.19788
[55]	valid_0's binary_logloss: 4.19379
[56]	valid_0's binary_logloss: 4.18878
[57]	valid_0's binary_logloss: 4.18505
[58]	valid_0's binary_logloss: 4.18065
[59]	valid_0's binary_logloss: 4.17573
[60]	valid_0's binary_logloss: 4.17143
[61]	valid_0's binary_logloss: 4.16746
[62]	valid_0's binary_logloss: 4.16289
[63]	valid_0's binary_logloss: 4.15895
[64]	valid_0's binary_logloss: 4.15466
[65]	valid_0's binary_logloss: 4.15129
[66]	valid_0's binary_logloss: 4.14746
[67]	valid_0's binary_logloss: 4.14378
[68]	valid_0's binary_logloss: 4.14049
[69]	valid_0's binary_logloss: 4.13787
[70]	valid_0's binary_logloss: 4.13488
[71]	valid_0's binary_logloss: 4.13166
[72]	valid_0's binary_logloss: 4.12881
[73]	valid_0's binary_log

## 11. 模型对比导出到 Excel

In [12]:
model_compare.to_excel(f"{OUT}/04_models_compare.xlsx", index=False)
sc.scorecard_points().to_excel(f"{OUT}/04_scorecard_points.xlsx", index=False)
print('已保存模型对比与评分卡刻度')

已保存模型对比与评分卡刻度
